# Demande conditionnelle des facteurs avec contrôles en 2022

In [88]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

## Chargement des données 

In [89]:
finess = pd.read_csv("finess.csv", sep=";", encoding="latin-1")
sygen = pd.read_csv("SAE/2022/SYGEN_2022r.csv", sep=";", encoding="latin-1")
urg = pd.read_csv("SAE/2022/URGENCES2_2022r.csv", sep=";", encoding="latin-1")
filtre = pd.read_csv("SAE/2022/FILTRE_2022r.csv", sep=";", decimal = ",", encoding="latin-1")
hd = pd.read_csv("Hospidiag/hd2022.csv", sep=";", decimal = ",", encoding="latin-1")

## Nettoyage des données

In [90]:
data = sygen

In [91]:
sygen['FI'].nunique() == sygen.shape[0]


True

In [92]:
finess['FINESS'].nunique() == finess.shape[0]

True

In [93]:
data['FI'].nunique()

3988

Le Finess est bien un identifiant unique qui permet de fusionner les bases sans problème. 

In [94]:
finess = finess[["FINESS","Raison sociale","Statut Juridique"]]
finess = finess.rename(columns={"FINESS":"FI","Raison sociale":"RS","Statut Juridique":"Statut"})
finess['FI'] = finess['FI'].astype(str)
data = data.merge(finess, how="left", on = "FI")
data.shape

(3988, 210)

In [95]:
urg = urg[['FI','PASSU']]
urg = urg.groupby('FI').sum().reset_index()
data = data.merge(urg, on="FI", how="left")
data.shape

(3988, 211)

In [96]:
hd = hd.rename(columns={"finess":"FI_EJ"})
data = data.merge(hd, how="left", on="FI_EJ")
data.shape

(3988, 367)

In [97]:
filtre = filtre[["STATUT","FI","HEB_MED","HEB_CHIR","HEB_PERINAT","HEB_PSY","HEB_SSR", "HEB_SLD"]]
filtre = filtre[filtre['STATUT']=='DECLA']
data = data.merge(filtre, how="left", on="FI")

## Agrégation des personnels 

In [98]:
data['MED'] = data['EFFSAL_TOT'] + data['EFFLIB_TOT']
data['ADMIN'] = data['EFF_DIR'] + data['EFF_AUTADM']
data['IDE'] = data['EFF_INFSANSSPE']
data['AID'] = data['EFF_AID']

## Agrégation des séances et des séjours en psychiatrie

In [99]:
data['SEANCES'] = data['SEAN_HEMO_CENTRE'] + data['SEAN_CHIMIO'] + data['SEAN_RADIO']
data['PSY'] = data['SEJ_HTP_TOT'] + data['VEN_HDJ_TOT'] + data['VEN_HDN_TOT']
data['SSR'] = data['SEJHC_SSR']
data['SLD'] = data['ENT']

## Passage des variables en log

In [100]:
data["lURG"] = np.where(data["PASSU"] > 0, np.log(data["PASSU"]), 0)
data["lMCO_AMB"] = np.where(data["SEJHP_MCO"] > 0, np.log(data["SEJHP_MCO"]), 0)
data["lMCO_COMP"] = np.where(data["SEJHC_MCO"] > 0, np.log(data["SEJHC_MCO"]), 0)
data["lPSY"] = np.where(data["PSY"] > 0, np.log(data["PSY"]), 0)
data["lSEANCES"] = np.where(data["SEANCES"] > 0, np.log(data["SEANCES"]), 0)
data["lSSR"] = np.where(data["SSR"] > 0, np.log(data["SSR"]), 0)
data["lSLD"] = np.where(data["SLD"] > 0, np.log(data["SLD"]), 0)

data["lMED"] = np.where(data["MED"] > 0, np.log(data["MED"]), 0)
data["lIDE"] = np.where(data["IDE"] > 0, np.log(data["IDE"]), 0)
data["lAID"] = np.where(data["AID"] > 0, np.log(data["AID"]), 0)
data["lADMIN"] = np.where(data["ADMIN"] > 0, np.log(data["ADMIN"]), 0)

c:\Users\roman\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\roman\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\roman\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\roman\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\roman\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufun

## Création des variables de statut

On veut 4 statuts : CHU, petit hôpital public, privé lucratif et privé non lucratif. On a déjà public, privé lucratif et privé non lucratif. On veut donc distinguer petit hôpital public et CHU parmi les hôpitaux publics. Les petits hôpitaux publics (publics mais non CHU) seront notre référence. 

In [101]:
chu = " CHU|CHU "
CHU_data = data[data['RS'].str.contains(chu, na=False)]
data['CHU'] = data['FI'].isin(CHU_data['FI']).astype(int)

In [102]:
data['PL'] = (data['Statut'] == 'Privé lucratif').astype(int)
data['PNL'] = (data['Statut'] == 'Privé non lucratif').astype(int)

In [103]:
data.shape

(3988, 396)

## Création d'une variable de gravité moyenne dans le département ?

## Demande conditionnelle de facteurs avec contrôles

In [104]:
outputs = ["lURG", "lMCO_AMB", "lMCO_COMP", "lSSR", "lPSY","lSEANCES", "lSLD"]
controls = ["PNL", "PL", "CHU","HEB_MED","HEB_CHIR","HEB_PERINAT","HEB_PSY","HEB_SSR", "HEB_SLD", "A9"]
inputs = ["lMED", "lIDE", "lAID", "lADMIN"]

In [107]:
data['A9'].isna().sum()

np.int64(2808)

In [87]:
hd['A9'].isna().sum()/hd.shape[0]

np.float64(0.1483221476510067)

In [81]:
results = {}
dict = {}

for var in outputs:
    data[f"{var}_PNL"] = data[var] * data["PNL"]
    data[f"{var}_PL"] = data[var] * data["PL"]
    data[f"{var}_CHU"] = data[var] * data["CHU"]


for inp in inputs:
    X = data[outputs + controls + [f"{v}_PNL" for v in outputs] + [f"{v}_PL" for v in outputs] + [f"{v}_CHU" for v in outputs]]
    X = sm.add_constant(X)
    y = data[inp]
    
    model = sm.OLS(y, X, missing='drop').fit(cov_type="HC1")  # erreurs robustes
    results[inp] = model
    
    print("\n" + "="*60)
    print(f"Demande conditionnelle pour : {inp}")
    print(model.summary())
    
    print(inp)
    for c in ["PL","PNL","CHU"]:
        coef = model.params[c]
        pct = (np.exp(coef) - 1) * 100
        print(f"{c} → différence d'utilisation : {pct:.2f}%")
        dict[(inp,c)] = pct


Demande conditionnelle pour : lMED
                            OLS Regression Results                            
Dep. Variable:                   lMED   R-squared:                       0.401
Model:                            OLS   Adj. R-squared:                  0.398
Method:                 Least Squares   F-statistic:                     79.75
Date:                Fri, 13 Feb 2026   Prob (F-statistic):          8.24e-284
Time:                        18:02:47   Log-Likelihood:                -6810.7
No. Observations:                3987   AIC:                         1.367e+04
Df Residuals:                    3965   BIC:                         1.380e+04
Df Model:                          21                                         
Covariance Type:                  HC1                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const     

c:\Users\roman\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 37, but rank is 21
  warnings.warn('covariance of constraints does not have full '
c:\Users\roman\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 37, but rank is 21
  warnings.warn('covariance of constraints does not have full '


                            OLS Regression Results                            
Dep. Variable:                   lIDE   R-squared:                       0.636
Model:                            OLS   Adj. R-squared:                  0.634
Method:                 Least Squares   F-statistic:                     298.4
Date:                Fri, 13 Feb 2026   Prob (F-statistic):               0.00
Time:                        18:02:47   Log-Likelihood:                -5885.6
No. Observations:                3987   AIC:                         1.182e+04
Df Residuals:                    3965   BIC:                         1.195e+04
Df Model:                          21                                         
Covariance Type:                  HC1                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const             1.5115      0.034     44.958

c:\Users\roman\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 37, but rank is 21
  warnings.warn('covariance of constraints does not have full '
c:\Users\roman\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 37, but rank is 21
  warnings.warn('covariance of constraints does not have full '


                            OLS Regression Results                            
Dep. Variable:                   lAID   R-squared:                       0.664
Model:                            OLS   Adj. R-squared:                  0.662
Method:                 Least Squares   F-statistic:                     336.1
Date:                Fri, 13 Feb 2026   Prob (F-statistic):               0.00
Time:                        18:02:47   Log-Likelihood:                -5974.2
No. Observations:                3987   AIC:                         1.199e+04
Df Residuals:                    3965   BIC:                         1.213e+04
Df Model:                          21                                         
Covariance Type:                  HC1                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const             1.0011      0.030     33.210

In [77]:
dict

{('lMED', 'PL'): np.float64(-6.661338147750939e-14),
 ('lMED', 'PNL'): np.float64(-5.551115123125783e-14),
 ('lMED', 'CHU'): np.float64(-44.06184742036011),
 ('lIDE', 'PL'): np.float64(-2.220446049250313e-14),
 ('lIDE', 'PNL'): np.float64(-1.1102230246251565e-14),
 ('lIDE', 'CHU'): np.float64(21.167038129756797),
 ('lAID', 'PL'): np.float64(-2.220446049250313e-14),
 ('lAID', 'PNL'): np.float64(-1.1102230246251565e-14),
 ('lAID', 'CHU'): np.float64(34.81331806983307),
 ('lADMIN', 'PL'): np.float64(0.0),
 ('lADMIN', 'PNL'): np.float64(-2.220446049250313e-14),
 ('lADMIN', 'CHU'): np.float64(-42.849063320431014)}

In [ ]:
#OK : ajouter indic  HEB_MED, HEB_CHIR,HEB_PERINAT,HEB_PSY,HEB_SSR du fichier filtre (ils sont déjà en indic)

#OK: voir comment traiter USLD car beaucoup de données manquantes : si on l'inclut, ajouter indic ULSD HEB_SLD 
# -> ajouté USLD et indic malgré le manque de données

#OK: rajouter log output x statut juridique 

#OK: rajouter distiction CHU/petit hôpital public, mettre petits hopitaux publics en réf -> comment est ce qu'on fait vu 
#qu'on est au niveau du finess géographique : est ce que les unités appartenant à un CHU sont considérées comme CHU ? je dirais oui

#gros problème : avec le manque de sur l'indice de sévérité, on ne garde que environ 800 observations / 3998...
#est ce qu'on remplace par 0 quand c'est pas renseigné ? est ce qu'on approxime par la moyenne dans le département ?

#ajouter durée de séjour

#est ce qu'on distingue activité psychiatrique à temps complet et à temps partiel ? pour le moment agrégés dans PSY, 
#est ce que le temps partiel correspond à l'ambulatoire ? dans ce cas autant distinguer comme pour MCO

In [73]:
data['SEJHC_SSR'].count() / data.shape[0]


np.float64(0.4518555667001003)

In [74]:
data['ENT'].count() / data.shape[0]

np.float64(0.1444332998996991)